In [ ]:
https://huggingface.co/medicalai/ClinicalBERT

https://huggingface.co/emilyalsentzer/Bio_ClinicalBERT

In [3]:
from transformers import AutoTokenizer, AutoModel

In [ ]:
# should already be in the cache
tokenizer_cb = AutoTokenizer.from_pretrained("medicalai/ClinicalBERT")
model_cb = AutoModel.from_pretrained("medicalai/ClinicalBERT")

In [4]:
# more modern model
from transformers import AutoTokenizer, AutoModel
tokenizer_bcb = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
model_bcb = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
import duckdb
from pathlib import Path

DB_PATH = Path("../mimic4_note.db").resolve()

con = duckdb.connect(
    str(DB_PATH),
    read_only=True
)

In [9]:
nlp_subset_readmit_sectioned = con.sql("""
SELECT *
FROM nlp_subset_readmit_sectioned
""").df()

In [10]:
con.close()

In [11]:
for col in [
    "brief_hospital_course",
    "medication_reconciliation",
    "discharge_planning"
]:
    print(
        col,
        nlp_subset_readmit_sectioned[col]
        .str.split()
        .str.len()
        .describe()
    )

brief_hospital_course count    5000.000000
mean      332.834000
std       269.799733
min         0.000000
25%       152.000000
50%       279.000000
75%       456.000000
max      2687.000000
Name: brief_hospital_course, dtype: float64
medication_reconciliation count    5000.000000
mean      214.068800
std       130.793259
min         5.000000
25%       121.750000
50%       190.000000
75%       280.000000
max      1104.000000
Name: medication_reconciliation, dtype: float64
discharge_planning count    5000.000000
mean      204.202000
std       168.289151
min         4.000000
25%       101.000000
50%       153.000000
75%       252.000000
max      1570.000000
Name: discharge_planning, dtype: float64


Some patients have very long notes, so we will need to truncate them to fit into the model's maximum input length (512 token maximum for both Bert-based models).

In [12]:
import torch

# move model to GPU if available

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model_bcb.to(device)
model.eval()

text = nlp_subset_readmit_sectioned.loc[0, "text"]

tokens = tokenizer_bcb(
    text,
    truncation=True,
    padding="max_length",
    max_length=512,
    return_tensors="pt"
)

# move inputs to the same device as the model
tokens = {k: v.to(device) for k, v in tokens.items()}

with torch.no_grad():
    outputs = model(**tokens)

print(outputs.last_hidden_state.shape)

torch.Size([1, 512, 768])


# Generating Embeddings with Bio-ClinicalBERT

* create embeddings for each selected section of the note using Bio-ClinicalBERT
    * brief_hospital_course
    * medication_reconciliation
    * discharge_planning
* we will use the [CLS] token embedding as the representation for the entire note section


In [13]:
import numpy as np
import torch
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_bcb = model_bcb.to(device)
model_bcb.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(28996, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [14]:
def embed_texts(texts, tokenizer, model, batch_size=16, max_length=512):
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts.iloc[i:i+batch_size].fillna("").tolist()

        tokens = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )

        tokens = {k: v.to(device) for k, v in tokens.items()}

        with torch.no_grad():
            outputs = model(**tokens)

        # CLS embedding: one 768-d vector per note
        cls_embeddings = outputs.last_hidden_state[:, 0, :]

        all_embeddings.append(cls_embeddings.cpu().numpy())

    return np.vstack(all_embeddings)

In [15]:
section_cols = [
    "brief_hospital_course",
    "medication_reconciliation",
    "discharge_planning"
]

embeddings_by_section = {}

for col in section_cols:
    print(f"Embedding: {col}")
    embeddings_by_section[col] = embed_texts(
        nlp_subset_readmit_sectioned[col],
        tokenizer_bcb,
        model_bcb,
        batch_size=16
    )

Embedding: brief_hospital_course


  0%|          | 0/313 [00:00<?, ?it/s]

Embedding: medication_reconciliation


  0%|          | 0/313 [00:00<?, ?it/s]

Embedding: discharge_planning


  0%|          | 0/313 [00:00<?, ?it/s]

In [16]:
for col, emb in embeddings_by_section.items():
    print(col, emb.shape)

brief_hospital_course (5000, 768)
medication_reconciliation (5000, 768)
discharge_planning (5000, 768)


In [17]:
X_brief = embeddings_by_section["brief_hospital_course"]
X_meds = embeddings_by_section["medication_reconciliation"]
X_plan = embeddings_by_section["discharge_planning"]

y = nlp_subset_readmit_sectioned["readmit_30d"].values

X_concat = np.hstack([ # combine the embeddings from the three sections into a single feature matrix; unsure if this will perform better
    X_brief,
    X_meds,
    X_plan
])

print(X_concat.shape)

(5000, 2304)


In [18]:
np.savez_compressed(
    "bioclinicalbert_section_embeddings_readmit.npz",
    X_brief=X_brief,
    X_meds=X_meds,
    X_plan=X_plan,
    X_concat=X_concat,
    y=nlp_subset_readmit_sectioned["readmit_30d"].values,
    subject_id=nlp_subset_readmit_sectioned["subject_id"].values # the groups
)

Bio_ClinicalBERT converts each discharge note section from unstructured text into a dense numerical representation (an embedding). Rather than manually engineering features from the note, the model compresses the semantic and clinical information contained in the text into a 768-dimensional vector, where notes with similar clinical meaning tend to have similar representations.

The saved .npz file therefore serves as the machine learning feature matrix. Instead of storing the original text, it stores the Bio_ClinicalBERT embeddings for each note section (e.g., Brief Hospital Course, Medication Reconciliation, and Discharge Planning), along with the readmission outcome and patient identifiers needed for grouped train–test splitting. These embeddings can then be used directly as inputs to downstream models such as Logistic Regression or XGBoost for readmission prediction.

In [19]:
con.close()